# Fine‑Tuning T5 for Sentiment Classification ( IMDB)


### 1. Install and import dependencies

In [1]:
!pip -q install transformers datasets sentencepiece accelerate

In [2]:
import math
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import AutoTokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f'PyTorch: {torch.__version__}')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))
    print('Mixed precision (AMP) enabled by default on CUDA.')


PyTorch: 2.10.0+cpu
CUDA available: False


### 2. Configuration & reproducibility

In [3]:
# ---- Experiment config ----
MODEL_NAME = 't5-small'              # small version of T5 for faster training
TASK_PREFIX = 'imdb sentiment: '     # Task prompt: T5 expects a natural language instruction
MAX_SOURCE_LENGTH = 256              # truncate long reviews for speed
MAX_TARGET_LENGTH = 4                # 'positive' or 'negative'
BATCH_SIZE = 8
NUM_EPOCHS = 2                       # keep small for speed
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06                  # warmup percentage of total steps
GRAD_ACCUM_STEPS = 2                 # simulate larger batches on limited GPU
MAX_NEW_TOKENS = 3                   # generation length for classification
NUM_BEAMS = 1                        # greedy is fine for classification

# Subsampling for classroom runs (set to None to use full sets)
SUBSET_TRAIN = 4000
SUBSET_VALID = 2000

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using DEVICE:', DEVICE)

# Output directory for metrics/plots
OUT_DIR = Path('t5_imdb_runs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Label verbalizers
id2label = {0: 'negative', 1: 'positive'}
label2id = {v: k for k, v in id2label.items()}
LABEL_TEXTS = [id2label[0], id2label[1]]


Using DEVICE: cpu


### 3. Load the IMDB dataset (binary sentiment)

In this example we will be using the [*IMDB - Large Movie Review Dataset*](https://huggingface.co/datasets/stanfordnlp/imdb). This is a dataset for binary sentiment classification with 25,000 movie reviews for training, and 25,000 for testing. There is additional unlabeled data for use as well.

In [4]:
ds = load_dataset('imdb')
print(ds)

train_raw = ds['train']
valid_raw = ds['test']              # use test as validation for simplicity (demo)

if SUBSET_TRAIN is not None:
    train_raw = train_raw.shuffle(seed=SEED).select(range(min(SUBSET_TRAIN, len(train_raw))))
if SUBSET_VALID is not None:
    valid_raw = valid_raw.shuffle(seed=SEED).select(range(min(SUBSET_VALID, len(valid_raw))))

print('Train size:', len(train_raw), 'Valid size:', len(valid_raw))


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Train size: 4000 Valid size: 2000


### 4. Tokenization & preprocessing

In T5, all tasks are frame as text-to-text. A task prefix is added before the input text and labels are also given in text formeat. Therefore, in this case, the input to the model and the target to predict will be:
- *Input*: `"imdb sentiment: <review>"`
- *Target*: `"positive"` or `"negative"`

We use **T5** tokenizer to tokenize the dataset. We tokenize review texts truncating the input text to `MAX_SOURCE_LENGTH` and labels (`positive` or `negative`) to `MAX_SOURCE_LENGTH`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# THIS IS THE PRE-PROCESSING FOR GPT2 MODEL. HOW SHOULD WE MODIFIFY THE CODE FOR T5?

# Map labels to integers and back
id2label = {0: "negative", 1: "positive"}
label2id = {v: k for k, v in id2label.items()}


def preprocess_function(batch):
    enc = tokenizer(batch["text"], max_length=MAX_LENGTH, truncation=True, padding=False)
    enc["labels"] = batch["label"]
    return enc

# Preprocess training and validation sets by tokenizing, truncating the reviews to MAX_LENGTH and mapping labels to integers. 
# We also remove the original text and label columns since they are no longer needed after tokenization.
train_tokenized = train_raw.map(preprocess_function, batched=True, remove_columns=train_raw.column_names)
valid_tokenized = valid_raw.map(preprocess_function, batched=True, remove_columns=valid_raw.column_names)

print(train_tokenized)
print(valid_tokenized)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})


### 5. Data Loader
We pad to the longest sequence in each batch, both inputs and labels. In the labels, pad tokens are converted to **-100** so the loss ignores them.

In [ ]:
def collate_fn(features):
    input_batch = {
        'input_ids': [f['input_ids'] for f in features],
        'attention_mask': [f['attention_mask'] for f in features],
    }
    labels_batch = {'input_ids': [f['labels'] for f in features]}

    padded_inputs = tokenizer.pad(input_batch, padding=True, return_tensors='pt')
    padded_labels = tokenizer.pad(labels_batch, padding=True, return_tensors='pt')['input_ids']
    
    # HOW SHOULD WE HANDLE PAD TOKENS IN LABELS?
    # ADD CODE HERE
    
    
    padded_inputs['labels'] = padded_labels
    return padded_inputs

train_loader = DataLoader(train_tokenized, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_tokenized, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

len(train_loader), len(valid_loader)

(500, 250)

### 6. Model, optimizer, and scheduler

We will be using [`T5ForConditionalGeneration`](https://huggingface.co/docs/transformers/en/model_doc/t5#transformers.T5ForConditionalGeneration), for text-to-text generation. 

The specific checkpoint of the model is specified by the parametre `MODEL_NAME`. By default, we will use 't5-small', the smallest available GPT model with 60M parameters. You can also try larger versions 't5-base' or 't5-large' (see https://huggingface.co/collections/google/t5-release)

In [7]:
# We load the pretrained T5 model for conditional generation
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
t_total_steps = NUM_EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(WARMUP_RATIO * t_total_steps)

lr_scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=t_total_steps)

print(f'Total steps: {t_total_steps} | Warmup steps: {num_warmup_steps}')


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Total steps: 500 | Warmup steps: 30


### 7. Inference with the pre-trained model

We do inference for the first 10 elements in the validation set to check the performanca of the pre-trained model before fine-tuning.

In [8]:
model.eval()

# Build a batch from the first 10 validation samples for a forward pass
examples = valid_tokenized.select(range(10))
enc = tokenizer.pad(
    {
        "input_ids": [x["input_ids"] for x in examples],
        "attention_mask": [x["attention_mask"] for x in examples],
    },
    padding=True,
    return_tensors="pt",
)
enc = {k: v.to(DEVICE) for k, v in enc.items()}
gen = model.generate(
    input_ids=enc['input_ids'],
    attention_mask=enc['attention_mask'],
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=NUM_BEAMS,
)
decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)

for text, pred in zip(examples, decoded):
    print("Review:", tokenizer.decode(text["input_ids"], skip_special_tokens=True))
    print("Prediction:", pred)
    print("---")
    

Review: imdb sentiment: br />br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?br />br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I started crying and couldn't stop until long after the movie ended. Thank you Jane, Laura and Jocelyn, for bringing us such a wonderfully subtle and compassionate movie! Thank you cast, for being involved and portraying the characters with such depth and gentleness!br />br />I recognized the Angry sister; the Runaway sister and the sister in Denial. I recognized the Abusive Husband and why he was there and then the Father, oh oh the Father... all superbly played. I also recognized myself and this movie was an eye-opener, a relief, a chance to face my OWN truth and finally doing something about it
Prediction: the movie
---
Review: imdb sentiment: This is the latest entry in th

### 8. Decoding the output: label mapping 
For each example we try to generate a text label and parse it as `positive` or `negative`.
If the generation is ambiguous, we score both label candidates by computing the T5 loss when forcing the target to 'positive' vs 'negative', and choose the lower‑loss label. This makes evaluation robust.

In [10]:
# Pre-tokenize label candidates for scoring fallback
label_token_ids = [tokenizer([l], max_length=MAX_TARGET_LENGTH, truncation=True)['input_ids'][0] for l in LABEL_TEXTS]

# Given the generated text, parse it to determine the predicted label (1 for positive, 0 for negative, -1 for unknown/ambiguous)
def parse_label(text):
    t = text.strip().lower()
    if 'positive' in t and 'negative' not in t:
        return 1
    if 'negative' in t and 'positive' not in t:
        return 0
    # ambiguous / unknown
    return -1

# Fallback in case of ambiguous/unknown labels: choose the label with lower teacher-forced loss for each sample.
def score_labels_for_batch(model, tokenizer, input_ids, attention_mask):
    predictions = []
    for i in range(input_ids.size(0)):
        # Get a single sample from batch (keep batch dimension for model input)
        x_ids = input_ids[i : i + 1]
        x_mask = attention_mask[i : i + 1]
        losses = []
        # Score each label candidate and take the one with lowest loss as prediction
        for lab_ids in label_token_ids:
            lab = torch.tensor(lab_ids, dtype=torch.long, device=input_ids.device).unsqueeze(0)
            lab = lab.masked_fill(lab == tokenizer.pad_token_id, -100)
            # Performs a teacher-forced pass and computes the loss with respect to the candidate label. 
            out = model(input_ids=x_ids, attention_mask=x_mask, labels=lab)
            losses.append(float(out.loss.detach().cpu()))
        predictions.append(int(np.argmin(losses)))
    return predictions


def predict_labels(model, tokenizer, batch_inputs: Dict[str, torch.Tensor]) -> List[int]:
    # Try generation first
    gen = model.generate(
        input_ids=batch_inputs['input_ids'],
        attention_mask=batch_inputs['attention_mask'],
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
    )
    # Decode the generated output and parse to get initial prediction labels   
    decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
    preds = [parse_label(x) for x in decoded]

    # If any of the predicted lables are ambiguoys (-1), rely on fallback scoring for those positions
    if any(p == -1 for p in preds):
        fallback_idx = [i for i, p in enumerate(preds) if p == -1]
        if len(fallback_idx) > 0:
            sub_in = {k: v[fallback_idx] for k, v in batch_inputs.items() if k in ('input_ids', 'attention_mask')}
            fb = score_labels_for_batch(model, tokenizer, sub_in['input_ids'], sub_in['attention_mask'])
            for j, idx in enumerate(fallback_idx):
                preds[idx] = fb[j]
    return preds





### 9. Training loop

In [11]:
from sklearn.metrics import accuracy_score, f1_score

train_loss_history, valid_loss_history = [], []
valid_acc_history, valid_f1_history = [], []

best_valid_acc = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    pbar = tqdm(enumerate(train_loader, start=1), total=len(train_loader), desc=f'Epoch {epoch} [train]')

    optimizer.zero_grad(set_to_none=True)

    for step, batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(**batch)
        # Scales the loss before backpropagation so that gradient accumulation matches the magnitude of a larger effective batch
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()

        # Only update weights and step scheduler every GRAD_ACCUM_STEPS to simulate larger batch size
        if step % GRAD_ACCUM_STEPS == 0:
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            lr_scheduler.step()
  
       # For logging, we accumulate the loss scaled by GRAD_ACCUM_STEPS to reflect the effective batch size.   
        running_loss += loss.item() * GRAD_ACCUM_STEPS
        avg_loss = running_loss / step
        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})

    train_loss_history.append(avg_loss)

    # ---- Validation ----
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        pbar_val = tqdm(valid_loader, desc=f'Epoch {epoch} [valid]')
        for batch in pbar_val:
            labels = batch['labels']
            # Keep a copy of gold labels before -100 masking fix for metrics
            gold = labels.clone()
            gold[gold == -100] = tokenizer.pad_token_id
            gold_text = tokenizer.batch_decode(gold, skip_special_tokens=True)
            gold_ids = [label2id.get(t.strip().lower(), 0) for t in gold_text]

            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            outputs = model(**batch)
            val_loss += outputs.loss.item()

            # Predictions via generate (+ fallback scoring if needed)
            inputs_cpu = {'input_ids': batch['input_ids'], 'attention_mask': batch['attention_mask']}
            preds = predict_labels(model, tokenizer, inputs_cpu)

            all_preds.extend(preds)
            all_labels.extend(gold_ids)

    mean_val_loss = val_loss / max(1, len(valid_loader))
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    valid_loss_history.append(mean_val_loss)
    valid_acc_history.append(acc)
    valid_f1_history.append(f1)

    # Save histories
    np.save(OUT_DIR / 'train_loss.npy', np.array(train_loss_history))
    np.save(OUT_DIR / 'valid_loss.npy', np.array(valid_loss_history))
    np.save(OUT_DIR / 'valid_acc.npy', np.array(valid_acc_history))
    np.save(OUT_DIR / 'valid_f1.npy', np.array(valid_f1_history))

    print(f'Epoch {epoch}: train_loss={avg_loss:.4f} | valid_loss={mean_val_loss:.4f} | acc={acc:.3f} | F1={f1:.3f}')

    # Save best by accuracy
    if acc > best_valid_acc:
        best_valid_acc = acc
        save_dir = 't5-imdb-best'
        os.makedirs(save_dir, exist_ok=True)
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f'Saved new best model to: {save_dir}')


Epoch 1 [train]:   0%|          | 0/500 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 10. Plot metrics across epochs
We visualize **training/validation loss** and **validation accuracy/F1**.

In [ ]:
# Reload histories if needed
if 'train_loss_history' not in globals():
    train_loss_history = np.load(OUT_DIR / 'train_loss.npy').tolist()
    valid_loss_history = np.load(OUT_DIR / 'valid_loss.npy').tolist()
    valid_acc_history = np.load(OUT_DIR / 'valid_acc.npy').tolist()
    valid_f1_history = np.load(OUT_DIR / 'valid_f1.npy').tolist()

epochs = list(range(1, len(train_loss_history) + 1))

plt.figure(figsize=(6,4))
plt.plot(epochs, train_loss_history, label='Training loss')
plt.plot(epochs, valid_loss_history, label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss curves')
plt.legend(); plt.tight_layout();
plt.savefig(OUT_DIR / 'loss_curves.png', dpi=150)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(epochs, valid_acc_history, label='Validation Accuracy')
plt.plot(epochs, valid_f1_history, label='Validation F1 (macro)')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Validation Accuracy & F1')
plt.legend(); plt.tight_layout();
plt.savefig(OUT_DIR / 'acc_f1_curves.png', dpi=150)
plt.show()

print('Saved figures to:', OUT_DIR.resolve())


### 11. Inference with the fine-tuned model

We do inference for the first 10 elements in the validation set to check the performanca of the fine-tuned model.

In [ ]:
model.eval()

# Build a batch from the first 10 validation samples for a forward pass
examples = valid_tokenized.select(range(10))
enc = tokenizer.pad(
    {
        "input_ids": [x["input_ids"] for x in examples],
        "attention_mask": [x["attention_mask"] for x in examples],
    },
    padding=True,
    return_tensors="pt",
)
enc = {k: v.to(DEVICE) for k, v in enc.items()}
gen = model.generate(
    input_ids=enc['input_ids'],
    attention_mask=enc['attention_mask'],
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=NUM_BEAMS,
)
decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)

for text, pred in zip(examples, decoded):
    print("Review:", tokenizer.decode(text["input_ids"], skip_special_tokens=True))
    print("Prediction:", pred)
    print("---")
    